In [ ]:
!pip install transformers

In [ ]:
# !pip install "transformers[torch]"
from google.colab import files

uploaded = files.upload()

Saving samsum-train.csv to samsum-train.csv
Saving samsum-validation.csv to samsum-validation.csv


In [ ]:
import pandas as pd
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [ ]:
import pandas as pd
from transformers import T5Tokenizer ,Trainer , TrainingArguments , T5ForConditionalGeneration

In [ ]:
def dialogues():
    for i in range(5):
        print(train_data["dialogue"][i])
        print("  ")


In [ ]:
# Random Sampling
train_data = train_data.sample(n=4000 , random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500 , random_state=42).reset_index(drop=True)

In [ ]:
train_data.shape

(4000, 3)

# Data Pre-Processing

In [ ]:
import re
def clean_data(text):
    text = re.sub(r"\r\n"," ",text) # lines
    text = re.sub(r"\s+"," ",text) # spaces
    text = re.sub(r"<.*?>"," ",text) # html tags
    text.strip().lower()
    return text

In [ ]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

In [ ]:
dialogues()

Violet: hi! i came across this Austin's article and i thought that you might find it interesting Violet:   Claire: Hi! :) Thanks, but I've already read it. :) Claire: But thanks for thinking about me :)
  
Pat: So does anyone know when the stream is going to happen? Lou: Unfortunately, no, but would really like to. Kevin: I don't think I'd be interested in this. Pat: Y? Kevin: Seeing all the blood and internal organs makes me dizzy. Lou: So you're so gentle? Pat: C'mon! Srsly? Kevin: Yup. Had the same thing since I was a child. Lou: Maybe it's time to change it? Pat: Yeah! Give it a try!
  
Jane:   Jane: Whaddya think? Shona: This ur tinder profile thing? Jane: Yeah, I'm updating my profile tonite. Kinda nervoous though... :( Jane: What if i get another guy like John? o.O Shona: John was a dickhead Jane: preach sistah! Shona: anyhoo - this time I've got u :D No slimeballs for you Jane: Not again *shudders* Jane: You know he forgot my birthday??!! Shona: wanker
  
Adam: Do u have a map 

In [ ]:
train_data

,id,dialogue,summary
0,13811908,Violet: hi! i came across this Austin's articl...,Violet sent Claire Austin's article.
1,13716431,Pat: So does anyone know when the stream is go...,Pat and Lou are waiting for The stream but Kev...
2,13810214,Jane: Jane: Whaddya think? Shona: This ur ti...,Jane is updating her Tinder profile tonight an...
3,13729823,"Adam: Do u have a map of Paris? Tom: Yes, Why?...",Tom has a map of Paris.
4,13681400,"Frank: Hi, how's the family? Mike: great! Sam'...","Mike is happy, because Sam's moved out. Mike a..."
...,...,...,...
3995,13681041,Barry: hello buddy Michael: hey Barry: do you ...,Barry and Michael will watch football instead ...
3996,13818705,Karen: Hey Lisa. Larissa and me have recently ...,Karen and Larissa moved to Belgium and ask Lis...
3997,13821859,"Miles: Hey, guys, I'm so sorry, but I missed t...","Miles has missed the bus, so he may be 15 minu..."
3998,13812716,Emma: did you finish the book I gave you? Liam...,"Emma gave ""The First Fifteen Lives of Harry Au..."


# Tokenizer

In [ ]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [ ]:
def tokenize(data):
    inputs = tokenizer(data["dialogue"],padding="max_length",max_length=512,truncation=True)
    targets = tokenizer(data["summary"],padding="max_length",max_length=128,truncation=True)

    inputs["labels"] = targets["input_ids"] # token ids => add to input as labels
    return inputs

In [ ]:
train_dataset = train_data.apply(tokenize , axis=1).tolist()
val_dataset = val_data.apply(tokenize , axis = 1).tolist()

In [ ]:
print(train_dataset[0])
print(len(train_dataset[0]["input_ids"]))

{'input_ids': [28866, 10, 7102, 55, 3, 23, 764, 640, 48, 8513, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 28866, 10, 19542, 10, 2018, 55, 3, 10, 61, 1333, 6, 68, 27, 31, 162, 641, 608, 34, 5, 3, 10, 61, 19542, 10, 299, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

# Working with Our Model : Fine Tuning

In [ ]:
model = T5ForConditionalGeneration.from_pretrained("t5-small")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
import torch

# print(torch.cuda.is_available())
# print(torch.cuda.device_count())
# print(torch.cuda.get_device_name(0))

print("Torch Version:", torch.__version__)
print("CUDA Version:", torch.version.cuda)
print("CUDA Available:", torch.cuda.is_available())



Torch Version: 2.11.0+cu128
CUDA Version: 12.8
CUDA Available: True


In [ ]:
# !pip uninstall -y torch torchvision torchaudio

In [ ]:
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
# import sys
# !{sys.executable} -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128 -v

In [ ]:
import torch
import sys

print(torch.__version__)
print(torch.__file__)
print(sys.executable)

2.11.0+cu128
/usr/local/lib/python3.12/dist-packages/torch/__init__.py
/usr/bin/python3


In [ ]:
# Training Arguments

training_args = TrainingArguments(
    output_dir = "./results",

    num_train_epochs=6,
    weight_decay=0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps=500
)

In [ ]:
# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,4.224636,0.456420
2,0.476948,0.429355
3,0.448756,0.419977
4,0.434534,0.416575
5,0.425392,0.414863
6,0.421481,0.414484


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=1.071958033243815, metrics={'train_runtime': 1171.7428, 'train_samples_per_second': 20.482, 'train_steps_per_second': 2.56, 'total_flos': 3248203235328000.0, 'train_loss': 1.071958033243815, 'epoch': 6.0})

In [ ]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model/tokenizer_config.json',
 './saved_summary_model/tokenizer.json')

In [ ]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [ ]:
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue) # clean
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # tokenize
    inputs = tokenizer(
        dialogue,
        padding="max_length",
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    # generate the summary => token ids
    model.to(device)
    targets = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=150,
        num_beams=4,
        early_stopping=True
    )

    # decoded our output
    summary = tokenizer.decode(targets[0], skip_special_tokens=True) # EOS, SEP
    return summary

In [ ]:
test_dialogue = """
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""

test01 = """Alice: Hi Bob, are you coming to the meeting tomorrow?
Bob: Yes, I will be there at 10 AM.
Alice: Great. Please bring the project report.
Bob: Sure, I'll bring it.
Alice: Thanks."""
summary = summarize_dialogue(test_dialogue)

print("Summary: ", summary)

Summary:  AI adoption has significantly increased over the past few years. Experts are concerned about bias in AI models because they reflect the data they are trained on.


In [ ]:
print(model)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [ ]:
# !zip -r saved_summary_model.zip saved_summary_model

  adding: saved_summary_model/ (stored 0%)
  adding: saved_summary_model/model.safetensors (deflated 9%)
  adding: saved_summary_model/tokenizer.json (deflated 75%)
  adding: saved_summary_model/generation_config.json (deflated 30%)
  adding: saved_summary_model/config.json (deflated 63%)
  adding: saved_summary_model/tokenizer_config.json (deflated 82%)
  adding: saved_summary_model/.ipynb_checkpoints/ (stored 0%)


In [ ]:
# from google.colab import files

# files.download("saved_summary_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>